# Import bibliotek

In [25]:
import requests
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime
import json

from dotenv import load_dotenv
load_dotenv()

True

# Pobranie danych pogodowych z Rapid

In [2]:
RAPIDAPI_KEY = os.environ["RAPIDAPI_KEY"]

## Historyczne dane pogodowe

In [3]:
url = "https://meteostat.p.rapidapi.com/stations/nearby"

params = {"lat":"49.298031137338704","lon":"19.949911081709388"}

headers = {
	"x-rapidapi-key": RAPIDAPI_KEY,
	"x-rapidapi-host": "meteostat.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=params)

data = response.json()["data"]

In [4]:
station_ids = ["12625", "12650", "11934"]

In [5]:
dfs = []
for station_id in station_ids:
    url = "https://meteostat.p.rapidapi.com/stations/daily"
    
    params = {"station": station_id, "start":"2020-01-01", "end":"2025-01-01"}
    
    headers = {
    	"x-rapidapi-key": RAPIDAPI_KEY,
    	"x-rapidapi-host": "meteostat.p.rapidapi.com",
    	"Content-Type": "application/json"
    }
    
    response = requests.get(url, headers=headers, params=params)
    data = response.json()["data"]
    df = pd.DataFrame(data)
    df["station_id"] = station_id
    dfs.append(df)

In [6]:
df_api_data = df_api_data = pd.concat(dfs, axis=0).reset_index(drop=True)

In [7]:
df_api_data = df_api_data.replace({"12625": "Zakopane", "12650": "Kasprowy Wierch", "11934": "Poprad_Tatry"})

In [8]:
columns_dict_rename = {
    "tavg": "avg_temp",
    "tmin": "min_temp",
    "tmax": "max_temp",
    "prcp": "precipitation_total_mm",
    "snow": "max_snow_dept_mm",
    "wdir": "wind_direction",
    "wspd": "avg_wind_speed_km/h",
    "wpgt": "max_wind_speed",
    "pres": "pressure",
    "tsun": "sunshine_total_min",
    "station_id": "station_name"
}

In [9]:
df_api_data = df_api_data.rename(columns_dict_rename, axis=1)

In [10]:
df_api_data.to_csv("data/weather_history_for_eda.csv", index=False)

# Pobranie danych pogodowych z OpenWeather 

In [11]:
OPENWEATHERAPI_KEY = os.environ["OPENWEATHERAPI_KEY"]

In [12]:
latitudes = np.linspace(49.02363485270159, 49.30738984415243, 10)
longitudes = np.linspace(19.670628703304338, 20.37238035455597, 10)

## Aktualne dane pogodowe

In [13]:
def get_current_weather(lat, lon):
    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "lat": lat,
        "lon": lon,
        "appid": OPENWEATHERAPI_KEY,
        "units": "metric"
    }
    
    response = requests.get(url, params=params)
    return response.json()["main"]

In [14]:
def get_air_pollution(lat, lon):
    url = "http://api.openweathermap.org/data/2.5/air_pollution/forecast"
    
    params = {
        "lat": lat,
        "lon": lon,
        "appid": OPENWEATHERAPI_KEY
    }
        
    response = requests.get(url, params=params)
    return response.json()["list"][0]["components"]["pm10"]

In [ ]:
dfs = []
for lat in tqdm(latitudes):
    for lon in longitudes:
        data = get_current_weather(lat, lon)
        df = pd.DataFrame(data, index=[0])
        df["pm10"] = get_air_pollution(lat, lon)
        df["lat"] = lat
        df["lon"] = lon
        df["download_timestamp"] = datetime.now().strftime("%Y%m%d_%H%M%S")
        dfs.append(df)
        
df_api_openweather = pd.concat(dfs, ignore_index=True)

 70%|██████████████████████████████████████████████████████████                         | 7/10 [00:28<00:12,  4.23s/it]

In [17]:
cols_to_drop = ["temp_min", "temp_max", "sea_level", "grnd_level"]
df_api_openweather = df_api_openweather.drop(cols_to_drop, axis=1, errors=True)
df_api_openweather

,temp,feels_like,pressure,humidity,pm10,lat,lon
0,4.89,3.64,1011,89,8.44,49.023635,19.670629
1,4.75,3.46,1011,89,8.44,49.023635,19.748601
2,4.59,3.31,1011,89,9.20,49.023635,19.826574
3,0.63,-1.10,1011,89,9.20,49.023635,19.904546
4,0.60,-2.38,1011,93,9.20,49.023635,19.982518
...,...,...,...,...,...,...,...
95,1.78,0.34,1012,90,10.40,49.307390,20.060491
96,1.85,0.34,1012,90,10.40,49.307390,20.138463
97,1.46,-0.31,1012,90,11.53,49.307390,20.216436
98,1.98,-0.06,1012,89,11.53,49.307390,20.294408


In [31]:
header_state = not "weather_history.csv" in os.listdir("data")

In [33]:
df_api_openweather.to_csv("data/weather_history.csv", mode="a", header=header_state, index=False)

# Prognoza pogody 

In [19]:
def get_forecast(lat, lon):
    url = "http://api.openweathermap.org/data/2.5/forecast"

    params = {
        "lat": lat,
        "lon": lon,
        "appid": OPENWEATHERAPI_KEY,
        "units": "metric"
    }
    
    response = requests.get(url, params=params)
    response_json = response.json()

    temps = [item["main"]["temp"] for item in response_json["list"]]
    tmstps = [item["dt_txt"] for item in response_json["list"]]
    
    return {
        "lat": lat,
        "lon": lon,
        "temperatures": 
            dict(zip(tmstps, temps))
    }

In [27]:
json_filename = f'{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
forecast_data = []

for lat in tqdm(latitudes):
    for lon in longitudes:
        forecast_data.append(get_forecast(lat, lon))

with open(f"data/json/{json_filename}", "w", encoding="utf-8") as f:
    json.dump(forecast_data, f, indent=2)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:27<00:00,  2.73s/it]
